# Task 1 Character-Level GPT Demonstration ? Anshika Goel

This executed notebook demonstrates the trained from-scratch character-level GPT, including:

- configuration and architecture metadata;
- training and validation loss curves;
- checkpoint loading and finite-parameter verification;
- validation-batch evaluation;
- greedy and temperature-based generation;
- final quantitative metrics.

The implementation uses custom causal multi-head self-attention, causal masking, layer normalization, GELU, feed-forward networks, residual connections, and learned token and positional embeddings. It does not use a prebuilt Transformer or attention module.


In [1]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import torch
import yaml

search_start = Path.cwd().resolve()
repo_root = next(
    candidate
    for candidate in [search_start, *search_start.parents]
    if (
        candidate
        / "task1_llm"
        / "Anshika_Goel"
    ).is_dir()
)

member_root = (
    repo_root
    / "task1_llm"
    / "Anshika_Goel"
)
code_directory = member_root / "code"

if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))

from data import CharVocabulary
from model import CharacterGPT, GPTConfig
from train import set_reproducible_seed

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

{
    "repository_location": ".",
    "member_directory": "task1_llm/Anshika_Goel",
    "device": str(device),
    "gpu": (
        torch.cuda.get_device_name(0)
        if device.type == "cuda"
        else None
    ),
}


C:\Users\aradh\Desktop\DATA 266 Generative AI and LLM\Lab1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'repository_location': '.',
 'member_directory': 'task1_llm/Anshika_Goel',
 'device': 'cuda',
 'gpu': 'NVIDIA GeForce RTX 3060 Laptop GPU'}

In [2]:
config_path = member_root / "configs" / "gpt_char.yaml"
vocabulary_path = (
    member_root
    / "outputs"
    / "metrics"
    / "vocabulary.json"
)
history_path = (
    member_root
    / "outputs"
    / "metrics"
    / "training_history.json"
)
training_summary_path = (
    member_root
    / "outputs"
    / "metrics"
    / "training_summary.json"
)
evaluation_metrics_path = (
    member_root
    / "outputs"
    / "metrics"
    / "evaluation_metrics.json"
)

with config_path.open("r", encoding="utf-8") as handle:
    experiment_config = yaml.safe_load(handle)

with history_path.open("r", encoding="utf-8") as handle:
    training_history = json.load(handle)

with training_summary_path.open(
    "r",
    encoding="utf-8",
) as handle:
    training_summary = json.load(handle)

with evaluation_metrics_path.open(
    "r",
    encoding="utf-8",
) as handle:
    evaluation_metrics = json.load(handle)

vocabulary = CharVocabulary.load(vocabulary_path)
model_settings = experiment_config["model"]
data_settings = experiment_config["data"]
training_settings = experiment_config["training"]

{
    "experiment": experiment_config["experiment"]["name"],
    "vocabulary_size": vocabulary.size,
    "context_length": data_settings["sequence_length"],
    "transformer_blocks": model_settings["num_layers"],
    "attention_heads": model_settings["num_heads"],
    "model_dimension": model_settings["d_model"],
    "feed_forward_dimension": model_settings["d_ff"],
    "dropout": model_settings["dropout"],
    "epochs": training_settings["epochs"],
    "parameter_count": training_summary["parameter_count"],
}


{'experiment': 'anshika_char_gpt_v1',
 'vocabulary_size': 101,
 'context_length': 256,
 'transformer_blocks': 5,
 'attention_heads': 6,
 'model_dimension': 240,
 'feed_forward_dimension': 960,
 'dropout': 0.1,
 'epochs': 10,
 'parameter_count': 3557861}

In [3]:
epochs = [
    record["epoch"]
    for record in training_history
]
training_losses = [
    record["training_loss"]
    for record in training_history
]
validation_losses = [
    record["validation_loss"]
    for record in training_history
]

figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(
    epochs,
    training_losses,
    marker="o",
    label="Training loss",
)
axis.plot(
    epochs,
    validation_losses,
    marker="o",
    label="Validation loss",
)
axis.set_xlabel("Epoch")
axis.set_ylabel("Cross-entropy loss")
axis.set_title("Character GPT Training and Validation Loss")
axis.grid(True, alpha=0.3)
axis.legend()
figure.tight_layout()
plt.show()

{
    "first_epoch_training_loss": training_losses[0],
    "final_training_loss": training_losses[-1],
    "first_epoch_validation_loss": validation_losses[0],
    "final_validation_loss": validation_losses[-1],
    "validation_improved_every_epoch": all(
        later < earlier
        for earlier, later in zip(
            validation_losses,
            validation_losses[1:],
        )
    ),
}


C:\Users\aradh\AppData\Local\Temp\ipykernel_9696\2713372722.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{'first_epoch_training_loss': 1.7003814176177978,
 'final_training_loss': 0.7292673282814026,
 'first_epoch_validation_loss': 1.017306755065918,
 'final_validation_loss': 0.6937300154685974,
 'validation_improved_every_epoch': True}

In [4]:
checkpoint_path = (
    member_root
    / "checkpoints"
    / "best_model.pt"
)
validation_tensor_path = (
    repo_root
    / "task1_llm"
    / "data"
    / "processed"
    / "Anshika_Goel"
    / "validation_sequences.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

model = CharacterGPT(
    GPTConfig(**checkpoint["model_config"])
)
model.load_state_dict(checkpoint["model_state_dict"])

parameters_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
)

model = model.to(device)
model.eval()

validation_tensor = torch.load(
    validation_tensor_path,
    map_location="cpu",
    weights_only=True,
)

inputs = validation_tensor[:32, :-1].long().to(device)
targets = validation_tensor[:32, 1:].long().to(device)

with torch.no_grad():
    with torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=device.type == "cuda",
    ):
        batch_outputs = model(inputs, targets)

{
    "checkpoint_epoch": checkpoint["epoch"],
    "checkpoint_validation_loss": checkpoint[
        "validation_loss"
    ],
    "all_parameters_finite": parameters_finite,
    "validation_batch_shape": tuple(inputs.shape),
    "validation_batch_loss": float(
        batch_outputs["loss"].item()
    ),
    "validation_batch_loss_finite": bool(
        torch.isfinite(batch_outputs["loss"]).item()
    ),
}


{'checkpoint_epoch': 10,
 'checkpoint_validation_loss': 0.6937300154685974,
 'all_parameters_finite': True,
 'validation_batch_shape': (32, 256),
 'validation_batch_loss': 0.7034752368927002,
 'validation_batch_loss_finite': True}

In [5]:
prompt = "Once upon a time"
prompt_ids = torch.as_tensor(
    vocabulary.encode(prompt),
    dtype=torch.long,
)
prompt_batch = prompt_ids.unsqueeze(0).to(device)

set_reproducible_seed(
    int(training_settings["training_seed"]) + 1
)

greedy_output = model.generate(
    prompt_batch,
    max_new_characters=120,
    greedy=True,
)

sampled_output = model.generate(
    prompt_batch,
    max_new_characters=120,
    temperature=1.0,
    greedy=False,
)

print("Greedy output:")
print(
    vocabulary.decode(
        greedy_output[0].detach().cpu().tolist()
    )
)

print("\nTemperature-1.0 sampled output:")
print(
    vocabulary.decode(
        sampled_output[0].detach().cpu().tolist()
    )
)


Greedy output:
Once upon a time, there was a little girl named Lily. She loved to play outside in the sunshine. One day, she went to the park with her 

Temperature-1.0 sampled output:
Once upon a time, there was a little girl named Lily. She loved to wave at the toy park. One day, she found a bath with a shiny rock tha


In [6]:
validation_metrics = evaluation_metrics["validation"]
generation_metrics = evaluation_metrics["generation"]
diversity_metrics = (
    evaluation_metrics["diversity"]["all_sampled_outputs"]
)

{
    "training_cross_entropy": round(
        training_summary["final_training_loss"],
        6,
    ),
    "validation_cross_entropy": round(
        validation_metrics["cross_entropy_loss"],
        6,
    ),
    "perplexity": round(
        validation_metrics["perplexity"],
        6,
    ),
    "bits_per_character": round(
        validation_metrics["bits_per_character"],
        6,
    ),
    "generalization_gap": round(
        validation_metrics["generalization_gap"],
        6,
    ),
    "top1_accuracy": round(
        validation_metrics["top1_accuracy"],
        6,
    ),
    "distinct_1": round(
        diversity_metrics["distinct_1"],
        6,
    ),
    "distinct_2": round(
        diversity_metrics["distinct_2"],
        6,
    ),
    "distinct_3": round(
        diversity_metrics["distinct_3"],
        6,
    ),
    "repeated_4gram_rate": round(
        diversity_metrics["repeated_4gram_rate"],
        6,
    ),
    "training_characters_per_second": round(
        training_summary["training_tokens_per_second"],
        2,
    ),
    "generation_characters_per_second": round(
        generation_metrics["tokens_per_second"],
        2,
    ),
    "peak_gpu_allocated_mb": round(
        training_summary["peak_gpu_allocated_mb"],
        2,
    ),
    "total_training_minutes": round(
        training_summary["total_training_seconds"] / 60.0,
        2,
    ),
}


{'training_cross_entropy': 0.729267,
 'validation_cross_entropy': 0.69373,
 'perplexity': 2.001166,
 'bits_per_character': 1.000841,
 'generalization_gap': -0.035537,
 'top1_accuracy': 0.780337,
 'distinct_1': 0.00437,
 'distinct_2': 0.039709,
 'distinct_3': 0.158188,
 'repeated_4gram_rate': 0.222371,
 'training_characters_per_second': 80067.67,
 'generation_characters_per_second': 518.08,
 'peak_gpu_allocated_mb': 1586.03,
 'total_training_minutes': 55.09}

## Supporting evidence

Complete evidence is retained in:

- `../results.md`
- `../failure_analysis.md`
- `../logs/training_run_001.log`
- `../logs/evaluation_generation_run_001.log`
- `../outputs/metrics/training_summary.json`
- `../outputs/metrics/evaluation_metrics.json`
- `../outputs/samples/generated_samples.json`
